In [ ]:
import pandas as pd
import numpy as ns
import seaborn as sns

In [ ]:
df = pd.read_csv("../dados/bruto/streaming_retencao.csv")

df.info()
df.head()

## Identificação de valores nulos

Verificação da quantidade de valores ausentes em cada coluna.

In [ ]:
df.isnull().sum()

## Tratamento da coluna `idade_do_usuário`

Foi identificada uma coluna chamada `idade_do_usuÃ¡rio`, contendo apenas 20 valores preenchidos. Além do nome inconsistente, seus valores (-15 e 150) não representam idades válidas, indicando um provável erro no dataset.

Por esse motivo, essa coluna será removida.

In [ ]:
df["idade_do_usuário"].dropna()

In [ ]:
df = df.drop(columns=["idade_do_usuário"])

## Tratamento da coluna `plano_atual`

A coluna `plano_atual` apresentou 30 valores nulos, correspondendo a aproximadamente 1,2% do conjunto de dados. Como não há informações que permitam identificar corretamente qual era o plano desses usuários, optou-se por remover esses registros. A quantidade de linhas excluídas é pequena em relação ao tamanho total do dataset, reduzindo o risco de impacto significativo nas análises e no treinamento do modelo.

In [ ]:
df["plano_atual"].value_counts(dropna=False)

In [ ]:
df = df.dropna(subset=["plano_atual"])

## Tratamento da coluna `genero_favorito`

A coluna `genero_favorito` apresentou 40 valores nulos, correspondendo a aproximadamente 1,6% do conjunto de dados. Como a ausência dessa informação não significa que o cliente pertença a outro gênero de conteúdo, optou-se por substituir os valores nulos pela categoria `Desconhecido`. Dessa forma, os registros são preservados sem atribuir um gênero incorreto aos usuários.

In [ ]:
df["genero_favorito"].value_counts(dropna=False)

In [ ]:
df["genero_favorito"] = df["genero_favorito"].fillna("Desconhecido")

df["genero_favorito"].value_counts(dropna=False)

## Verificação

Após os tratamentos realizados, foi feita uma nova verificação dos valores nulos.

In [ ]:
df.isnull().sum()

## Identificação e remoção de registros duplicados

Foi realizada a verificação de registros duplicados no dataset. Nenhuma linha duplicada foi identificada, portanto não foi necessário realizar nenhuma exclusão nesta etapa.

In [ ]:
df.duplicated().sum()

## Renomeando Colunas
Foram renomeadas algumas colunas para correção de erros ortográficos e simplificação

In [ ]:
df.tail()

In [ ]:
df = df.rename(columns = {'camcelou': 'cancelou', 'metodo_de_pagamento': 'pagamento', 'plano_atual': 'plano'})

## Remoção de Colunas
Foram removidas as colunas de identificação

In [ ]:
df = df.drop(columns = ['id', 'nome_do_cliente'])

## Análise e Tratamento de Outliers
Foi feita a contagem e análise da possível relação de outliers com um comportamento do dataset, caso fossem realmente valores atípicos sem relação, foram removidos

In [ ]:
df.tail()

## Análise de variáveis discretas
Tickets: Limite Superior = 5, valor max = 6, qtd outliers = 1, cancelou = Não (esperado para 6 tickets abertos seria o cancelamento) -> decisão: remoção

Séries abandonadas: Limite Superior = 7, valor max = 10, qtd outliers = 30, cancelou = 27 Não, 3 Sim  (o esperado seria uma quantidade maior de cancelamentos, então a presença destes dados pode comprometer) -> decisão: remoção

Séries concluidas: Limite Superior = 13, qtd outliers = 2, cancelou = ambos Não (é esperado para quem assistiu tantas séries) -> decisão: manter

In [ ]:
# Valores únicos
print(f"tickets_values: {sorted([int(x) for x in df['tickets_suporte_abertos'].unique()])}")
print(f"abandonadas_values: {sorted([int(x) for x in df['series_abandonadas'].unique()])}")
print(f"concluidas_values: {sorted([int(x) for x in df['series_concluidas'].unique()])}")   

In [ ]:
# Remoção outliers discretos selecionados
df = df[df['tickets_suporte_abertos'] < 6]
df = df[df['series_abandonadas'] < 8]

## Análise de Variáveis contínuas
Último acesso: Limite Superior = 44, qtd outliers = 118, cancelou = 91 Não e 27 Sim (muito tempo inativo seria esperado o cancelamento, 27 pra 91 é uma boa proporção considerando a dinâmica vista no dataset) -> decisão: manter até o último Sim (87)

Horas consumidas: Limite Superior = 254.5, qtd outliers = 4, cancelou = todos Não (esperado para quem usa tanto o serviço) -> decisão manter; obs: notou-se a existência de valores negativos, o que é logicamente inesperado, logo foram removidos

In [ ]:
# Análise da coluna de último acesso
coluna = df['dias_desde_ultimo_acesso']

Q1 = coluna.quantile(0.25)
Q3 = coluna.quantile(0.75)
IQR = Q3 - Q1

li = Q1 - 1.5 * IQR
ls = Q3 + 1.5 * IQR

outliers = coluna[(coluna < li) | (coluna > ls)]

print(f"Limites: {li:.2f} a {ls:.2f}")
print(f"Quantidade de outliers: {len(outliers)}")
print(f"Valores outliers:\n{outliers}")

In [ ]:
# Filtragem para o últim valor que cancelou (87)
df = df[df['dias_desde_ultimo_acesso'] < 88]

In [ ]:
# Análise da coluna horas consumidas
coluna = df['horas_consumidas']

Q1 = coluna.quantile(0.25)
Q3 = coluna.quantile(0.75)
IQR = Q3 - Q1

li = Q1 - 1.5 * IQR
ls = Q3 + 1.5 * IQR

outliers = coluna[(coluna < li) | (coluna > ls)]

print(f"Limites: {li:.2f} a {ls:.2f}")
print(f"Quantidade de outliers: {len(outliers)}")
print(f"Valores outliers:\n{outliers}")

In [ ]:
# Filtragem dos valores negativos
df = df[df['horas_consumidas'] >= 0]

## Análise de  Inconsistências
Verificação de valores anormais e/ou logicamente indevidos nas variáveis

In [ ]:
df.tail()

In [ ]:
# Não há menores de 18 anos -> OK
df[df['idade_do_usuario'] < 18]

In [ ]:
# Não há valores inesperados -> OK
print(df['cancelou'].unique())
print(df['plano'].unique())
print(df['pagamento'].unique())
print(df['dispositivo_mais_acesso'].unique())
print(df['genero_favorito'].unique())

# Exportar CSV

In [ ]:
df.to_csv("../dados/processado/streaming_retencao_limpo.csv", index=False)